In [3]:
import earthaccess
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

# Authenticate using the ~/.netrc file
auth = earthaccess.login()

# Define your Gadi Scratch directory for storage (Home is too small!)
output_dir = "/g/data/k10/zr7147/GPM_IMERG_Data"
os.makedirs(output_dir, exist_ok=True)

print(f"Authenticated. Data will be saved to: {output_dir}")

Authenticated. Data will be saved to: /g/data/k10/zr7147/GPM_IMERG_Data


In [4]:
# Define ORCESTRA region: [Lon Min, Lat Min, Lon Max, Lat Max]
bbox = (-65, 5, -15, 20) 

# The campaign Date Range (YYYY-MM-DD)
date_range = ("2024-08-10", "2024-09-30")

# Search for GPM IMERG Final Run (Half-Hourly)
results = earthaccess.search_data(
    short_name="GPM_3IMERGHH",
    bounding_box=bbox,
    temporal=date_range
)

print(f"Found {len(results)} half-hourly files for these dates.")

# Download only the files we don't already have
downloaded_files = earthaccess.download(results, local_path=output_dir)

Found 2496 half-hourly files for these dates.


QUEUEING TASKS | :   0%|          | 0/2496 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2496 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/2496 [00:00<?, ?it/s]

In [5]:
# Path setup
input_pattern = os.path.join(output_dir, "*.HDF5") 
output_path = "/g/data/k10/zr7147/ORCESTRA_IMERG_Combined_Cropped.nc"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [6]:
def crop_and_fix(ds):
    return ds.sel(lon=slice(-65, -15), lat=slice(5, 20))

In [11]:
from dask.distributed import Client

# Initialize the 16 workers you requested in your PBS script
# memory_limit is per worker (32GB / 16 = 2GB per worker)
client = Client(n_workers=16, threads_per_worker=1, memory_limit='2GB')
print(f"Dask is ready! View dashboard at: {client.dashboard_link}")

/g/data/k10/zr7147/orcestra_env/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43489 instead
  warnings.warn(


Dask is ready! View dashboard at: http://127.0.0.1:43489/status


In [12]:
ds_gpm = xr.open_mfdataset(
    input_pattern, 
    concat_dim='time', 
    combine='nested', 
    engine='h5netcdf', 
    group='/Grid',
    chunks={'time': 1}, # One chunk per file for faster merging
    parallel=True
)

# 3. Only sort if absolutely necessary!
# ds_gpm = ds_gpm.sortby('time') 

# 4. Compression (Keep this, it's good!)
encoding = {var: {'zlib': True, 'complevel': 4} for var in ds_gpm.data_vars}

print(f"Processing in parallel using {len(client.scheduler_info()['workers'])} workers...")

# 5. Write
ds_gpm.to_netcdf(output_path, encoding=encoding)
print("Success!")

/scratch/nf33/zr7147/tmp/ipykernel_243882/1599296457.py:1: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_gpm = xr.open_mfdataset(


Processing in parallel using 5 workers...


PermissionError: [Errno 13] Permission denied: '/g/data/k10/zr7147/ORCESTRA_IMERG_Combined_Cropped.nc'

In [ ]:
# Open with Parallel Dask Processing
# chunks={'lon': 400, 'lat': 400} tells xarray to process the map in tiles
ds_gpm = xr.open_mfdataset(
    input_pattern, 
    concat_dim='time', 
    combine='nested', 
    engine='h5netcdf', 
    group='/Grid',
    chunks={'time': 1,'lon': 400, 'lat': 400}, 
    parallel=True
)

ds_gpm = ds_gpm.sortby('time')

# 4. Save with Compression (saves disk space on Gadi!)
encoding = {var: {'zlib': True, 'complevel': 4} for var in ds_gpm.data_vars}

# Physical Write to Disk
# We use 'netcdf4' engine and 'least_significant_digit' to compress if needed
print(f"Starting the combined write to: {output_path}...")
print("This will process files one-by-one to keep RAM usage low.")

ds_gpm.to_netcdf(
    output_path, 
    mode='w', 
    format='NETCDF4', 
    compute=True
)

print("Success! Single cropped file created.")

/g/data/k10/zr7147/orcestra_env/lib/python3.12/site-packages/dask/_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 400. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
/g/data/k10/zr7147/orcestra_env/lib/python3.12/site-packages/dask/_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "lat" starting at index 400. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
/g/data/k10/zr7147/orcestra_env/lib/python3.12/site-packages/dask/_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 400. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
/g/data/k10/zr7147/orcestra_env/lib/python3.12/site-packages/dask/_tas

KeyboardInterrupt: 

In [ ]:
# Load all downloaded files into one dataset
file_pattern = os.path.join(output_dir, "*.HDF5") 
ds_gpm = xr.open_mfdataset(file_pattern, concat_dim='time', combine='nested', engine='h5netcdf', group='/Grid')

# Extract 'precipitation' (Calibrated Precipitation in mm/hr)
# Note: IMERG longitude is -180 to 180. We slice to our flight region.
precip = ds_gpm['precipitation'].sel(lon=slice(-65, -15), lat=slice(5, 20))
print("Precipitation data loaded. Dataset info:")
print(precip.dims)

KeyboardInterrupt: 